# 7 · Representative Propeller Selection

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

Selects a single unified subset of **100 propellers** for physical validation. Every selected prop is printed once and measured on the available rigs: scale (mass), trifilar pendulum (Izz), and the free-flight launcher (h_max, flight time, impact speed). Selection uses greedy k-centre (max-min distance) over six geometry features so the subset spans the design space evenly, stratified into two bands: Band 0 (8 props) covers the near-boundary non-liftoff cases with T/W ∈ [0.70, 1.00), where a systematic thrust over-prediction would flip the most consequential binary prediction the simulation makes; Band 1 (92 props) covers the liftoff population, split across 4 equal-quantile h_max tiers (23 each) so the high-flier tail stays represented.

**Physics inputs:** `csv/06_flight_dynamics.csv` (reference launch RPM), `csv/01_geometry.csv`, `csv/03_mass_inertia.csv`, `csv/05_qprop_results.csv`

**Physics outputs:** `csv/07_selected.csv`, `csv/07_all_subsets.csv` (the selected subset with its band/tier labels and key parameters), `plots/nb7/07_selection_coverage.png`, and the copied meshes in `representative_stl/`

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — greedy k-centre selection and the check helper
4. Main code — pool construction, Band 0 and Band 1 selection, validation, coverage visualisation, export and STL copying

> **Note:** this notebook always uses the reference-RPM flight dynamics (4000 RPM), which is below the QProp sweep ceiling and therefore unaffected by sweep-coverage limits.

> **Warning — do not overwrite the historical selection.** The shipped `csv/07_selected.csv` records the 100 propellers that were **actually printed and physically tested**. It was generated before the NB6 body-drag model revision, so re-running this notebook on the current flight-dynamics outputs produces a *different* (equally valid) subset and **overwrites the file**. Re-run only if you intend to select a new batch for fabrication; otherwise keep the shipped file, since NB9 and the validated dataset depend on it matching the printed parts.

> **Dead-code audit (2026-07-04):** all functions, variables and imports in this notebook were checked for use; nothing unused was found — no removals.


## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

import shutil
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

project_root = Path('.')
if str(project_root.resolve()) not in sys.path:
    sys.path.insert(0, str(project_root.resolve()))
import pipeline_config as cfg

## 2. Configuration

In [ ]:
N_BAND0 = cfg.N_BAND0
N_BAND1 = cfg.N_BAND1
N_TIERS = cfg.N_TIERS

TW_LO = cfg.TW_LO
TW_HI = cfg.TW_HI

GEO_FEATURES = cfg.GEO_FEATURES
RANDOM_SEED = cfg.RANDOM_SEED

ROOT = Path('.')
CSV_DIR = ROOT / 'csv'
OUT_CSV = CSV_DIR / cfg.CSV_NAMES['selected']
PLOTS_DIR = ROOT / 'plots' / 'nb7'
OUT_ALL = CSV_DIR / cfg.CSV_NAMES['all_subsets']

rng = np.random.default_rng(RANDOM_SEED)

print(f'Band 0 : {N_BAND0} non-liftoff props  (T/W {TW_LO}–{TW_HI})')
print(f'Band 1 : {N_BAND1} liftoff props  ({N_TIERS} tiers × {N_BAND1 // N_TIERS} each)')
print(f'Total  : {N_BAND0 + N_BAND1}')

## 3. Function Definitions

- **greedy_maxmin(coords, n, seed_rng)** — picks a spread-out subset of designs: starting from a random seed design, it repeatedly adds the design farthest (max-min distance) from everything chosen so far, which guarantees a covering radius within a factor of two of the optimum. Takes the normalised feature coordinates and the subset size; returns the selected row indices.
- **select_from_band(df, n, features, seed_rng)** — applies the greedy selection to one band: normalises the chosen geometry features to [0, 1], runs greedy_maxmin, and returns the selected rows.
- **chk(cond, msg)** — pass/fail assertion helper used to verify the combined selection meets the band and tier requirements.


In [ ]:
def greedy_maxmin(coords, n, seed_rng):

    M = len(coords)
    if n >= M:

        return list(range(M))

    first = int(seed_rng.integers(M))
    selected = [first]
    min_dists = cdist(coords, coords[[first]]).ravel()
    for iteration_index in range(n - 1):
        nxt = int(np.argmax(min_dists))
        selected.append(nxt)
        min_dists = np.minimum(min_dists, cdist(coords, coords[[nxt]]).ravel())

    return selected


def select_from_band(df, n, features, seed_rng):

    df = df.copy().reset_index(drop=True)
    if n >= len(df):

        return df

    norm_cols = []
    for col in features:
        if col not in df.columns:
            continue
        lo = df[col].min()
        hi = df[col].max()
        df[col + '_n'] = (df[col] - lo) / (hi - lo + 1e-12)
        norm_cols.append(col + '_n')

    coords = df[norm_cols].fillna(0).values
    indices = greedy_maxmin(coords, n, seed_rng)

    return df.iloc[indices].copy()


def chk(cond, msg):

    global all_ok
    print(f'  [{"PASS" if cond else "FAIL"}]  {msg}')
    if not cond:
        all_ok = False

## 4. Main Code

The main code loads the pipeline outputs, builds the eligible pool (valid STL and valid QProp prediction), selects Band 0 from the near-boundary non-liftoff window and Band 1 across the four h_max tiers, validates the combined subset, visualises its design-space coverage, saves the selection tables and copies the selected STLs to `representative_stl/`.


### 4.1 Load Data and Build the Eligible Pool

In [ ]:
geo = pd.read_csv(CSV_DIR / cfg.CSV_NAMES['geometry'])
stl = pd.read_csv(CSV_DIR / cfg.CSV_NAMES['mass_inertia'])
qres = pd.read_csv(CSV_DIR / cfg.CSV_NAMES['qprop_results'])
dyn = pd.read_csv(CSV_DIR / cfg.CSV_NAMES['flight_dynamics'])

if 'izz_regressed' in stl.columns:
    stl_columns = stl[['config_id', 'stl_ok', 'izz', 'izz_regressed']]
else:
    stl_columns = stl[['config_id', 'stl_ok', 'izz']]

master = dyn.merge(geo, on='config_id', how='left').merge(stl_columns, on='config_id', how='left').merge(qres[['config_id', 'qprop_ok']], on='config_id', how='left')

master['qprop_ok'] = master['qprop_ok'].fillna(False).astype(bool)
master['stl_ok'] = master['stl_ok'].fillna(False).astype(bool)
master['mass_g'] = master['mass_kg'] * 1000.0

pool = master[(master['stl_ok'] == True) & (master['qprop_ok'] == True) & (master['T_over_W'].notna())].copy().reset_index(drop=True)

print(f'Master frame : {len(master)} configs')
print(f'Eligible pool: {len(pool)} configs  (stl_ok=True AND qprop_ok=True)')
print(f'  of which can_liftoff=True : {pool["can_liftoff"].sum()}')
print(f'  of which can_liftoff=False: {(~pool["can_liftoff"]).sum()}')

### 4.2 Band 0 — Near-Boundary Non-Liftoff

Props with T/W < 0.7 are trivially non-flying, so validating there teaches nothing; the window T/W ∈ [0.7, 1.0) is the decision boundary where a systematic thrust over-prediction would incorrectly classify a prop as flyable.


In [ ]:
band0_pool = pool[(pool['can_liftoff'] == False) & (pool['T_over_W'] >= TW_LO) & (pool['T_over_W'] < TW_HI)].copy()

print(f'Band 0 pool : {len(band0_pool)} eligible configs')
print(f'  T/W range : {band0_pool["T_over_W"].min():.3f} – {band0_pool["T_over_W"].max():.3f}')

band0 = select_from_band(band0_pool, N_BAND0, GEO_FEATURES, rng)
band0['band'] = 0
band0['h_max_tier'] = -1
band0['subset'] = 'band0_nonliftoff'

print(f'\nBand 0 selected: {len(band0)} configs')
print(band0[['config_id', 'T_over_W', 'T_static_N', 'mass_g', 'radius_mm', 'blade_count']].sort_values('T_over_W').to_string(index=False))

### 4.3 Band 1 — Liftoff Tiers

h_max, flight_time and v_impact are r = 0.93–0.98 correlated, so stratifying on h_max distributes coverage across all three flight outputs simultaneously. Without tiers, greedy k-centre on geometry alone would cluster in the dense 1–2 m region and leave the high-flier tail unrepresented.


In [ ]:
used_ids = set(band0['config_id'])
band1_pool = pool[(pool['can_liftoff'] == True) & (pool['h_max_m'] > 0) & (~pool['config_id'].isin(used_ids))].copy().reset_index(drop=True)

print(f'Band 1 pool: {len(band1_pool)} eligible liftoff configs')

tier_edges = np.quantile(band1_pool['h_max_m'], np.linspace(0, 1, N_TIERS + 1))
tier_edges[0] -= 1e-9
tier_edges[-1] += 1e-9

print('\nTier edges (h_max quantiles):')
for i in range(N_TIERS):
    print(f'  Tier {i}: {tier_edges[i]:.3f} – {tier_edges[i + 1]:.3f} m')

band1_pool['h_max_tier'] = pd.cut(band1_pool['h_max_m'], bins=tier_edges, labels=list(range(N_TIERS)), include_lowest=True).astype(int)

budget_per_tier = N_BAND1 // N_TIERS
remainder = N_BAND1 - budget_per_tier * N_TIERS
tier_budgets = {}
for i in range(N_TIERS):
    tier_budgets[i] = budget_per_tier
for i in range(remainder):
    tier_budgets[N_TIERS - 1 - i] += 1

print(f'\nTier budgets: {tier_budgets}')

tier_pieces = []
for tier_idx in range(N_TIERS):
    tier_pool = band1_pool[band1_pool['h_max_tier'] == tier_idx].copy()
    n_select = tier_budgets[tier_idx]
    sel = select_from_band(tier_pool, n_select, GEO_FEATURES, rng)
    sel['band'] = 1
    sel['h_max_tier'] = tier_idx
    sel['subset'] = f'band1_tier{tier_idx}'
    tier_pieces.append(sel)
    print(f'  Tier {tier_idx}: pool={len(tier_pool)}, selected={len(sel)}, h_max=[{tier_pool["h_max_m"].min():.3f}, {tier_pool["h_max_m"].max():.3f}] m')

band1 = pd.concat(tier_pieces, ignore_index=True)
print(f'\nBand 1 total selected: {len(band1)}')

### 4.4 Combine and Validate

In [ ]:
selected = pd.concat([band0, band1], ignore_index=True)

assert selected['config_id'].is_unique, 'FAIL: duplicate config_ids in selection'

b0 = selected[selected['band'] == 0]
b1 = selected[selected['band'] == 1]

print(f'Total selected : {len(selected)} configs  (target: {N_BAND0 + N_BAND1})')
print()
print('Band 0 (non-liftoff):')
print(f'  n={len(b0)},  T/W: [{b0["T_over_W"].min():.3f}, {b0["T_over_W"].max():.3f}]')
print()
print('Band 1 (liftoff tiers):')
for t in range(N_TIERS):
    tt = b1[b1['h_max_tier'] == t]
    print(f'  Tier {t}: n={len(tt)},  h_max=[{tt["h_max_m"].min():.3f}, {tt["h_max_m"].max():.3f}] m,  T/W=[{tt["T_over_W"].min():.2f}, {tt["T_over_W"].max():.2f}]')
print()

all_ok = True

chk(len(selected) == N_BAND0 + N_BAND1, f'Total size = {N_BAND0 + N_BAND1}')
chk(selected['config_id'].is_unique, 'No duplicate config_ids')
chk(selected['stl_ok'].all(), 'All selected: stl_ok=True')
chk(selected['qprop_ok'].all(), 'All selected: qprop_ok=True')
chk((b0['can_liftoff'] == False).all(), 'Band 0: all can_liftoff=False')
chk((b0['T_over_W'] >= TW_LO).all() and (b0['T_over_W'] < TW_HI).all(), f'Band 0: all T/W in [{TW_LO}, {TW_HI})')
chk((b1['can_liftoff'] == True).all(), 'Band 1: all can_liftoff=True')
chk(b1['h_max_tier'].nunique() == N_TIERS, f'Band 1: all {N_TIERS} tiers represented')
chk(b1['T_over_W'].max() < 10.0, f'Band 1: T/W max plausible (<10), got {b1["T_over_W"].max():.2f}')

t_span_pool = pool['T_static_N'].max() - pool['T_static_N'].min()
t_span_sel = selected['T_static_N'].max() - selected['T_static_N'].min()
chk(t_span_sel / t_span_pool > 0.8, f'T_static coverage > 80% of pool range ({100 * t_span_sel / t_span_pool:.1f}%)')

print()
print('ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED — see above')

### 4.5 Coverage Visualisation

Shows how the selected subset covers the eligible pool in T/W, the h_max tiers, the mass–inertia plane, and the combined (T/W, h_max) space.


In [ ]:
BAND_COLORS = {0: '#e63946', 1: '#2a9d8f'}
TIER_COLORS = ['#264653', '#457b9d', '#2a9d8f', '#52b788']

fig = plt.figure(figsize=(18, 11))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(pool['T_over_W'].clip(0, 5), bins=60, color='#adb5bd', alpha=0.5, label='Eligible pool')
ax1.scatter(b0['T_over_W'], np.ones(len(b0)) * 5, color=BAND_COLORS[0], s=60, zorder=5, label=f'Band 0 non-liftoff (n={len(b0)})')
ax1.scatter(b1['T_over_W'], np.ones(len(b1)) * 5, color=BAND_COLORS[1], s=20, alpha=0.7, zorder=4, label=f'Band 1 liftoff (n={len(b1)})')
ax1.axvline(1.0, color='white', lw=1.2, ls='--', label='T/W = 1')
ax1.axvline(TW_LO, color=BAND_COLORS[0], lw=1, ls=':')
ax1.set_xlabel('T/W ratio at launch')
ax1.set_ylabel('Count')
ax1.set_title('T/W coverage')
ax1.legend(fontsize=7)

ax2 = fig.add_subplot(gs[0, 1])
all_edges = np.linspace(band1_pool['h_max_m'].min(), band1_pool['h_max_m'].max(), 20)
bar_width = (all_edges[1] - all_edges[0]) / (N_TIERS + 1)
for t in range(N_TIERS):
    tt = b1[b1['h_max_tier'] == t]
    counts, bin_edges = np.histogram(tt['h_max_m'], bins=all_edges)
    centres = 0.5 * (all_edges[:-1] + all_edges[1:])
    offset = (t - N_TIERS / 2 + 0.5) * bar_width
    ax2.bar(centres + offset, counts, width=bar_width * 0.9, color=TIER_COLORS[t], alpha=0.9, label=f'Tier {t} (n={len(tt)})')
ax2.set_xlabel('h_max [m]')
ax2.set_ylabel('Count')
ax2.set_title('h_max tier coverage (Band 1)')
ax2.legend(fontsize=7)

ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(pool['mass_kg'] * 1000, pool['izz'] * 1e4, color='#adb5bd', alpha=0.15, s=6, label='Pool')
ax3.scatter(b0['mass_kg'] * 1000, b0['izz'] * 1e4, color=BAND_COLORS[0], s=70, zorder=5, label='Band 0')
ax3.scatter(b1['mass_kg'] * 1000, b1['izz'] * 1e4, color=BAND_COLORS[1], s=30, alpha=0.7, zorder=4, label='Band 1')
ax3.set_xlabel('Mass [g]')
ax3.set_ylabel('Izz [×10⁻⁴ kg·m²]')
ax3.set_title('Mass vs Izz coverage')
ax3.legend(fontsize=7)

ax4 = fig.add_subplot(gs[1, :])
ax4.scatter(pool['T_over_W'], pool['h_max_m'].fillna(0), color='#adb5bd', alpha=0.12, s=6, label='Eligible pool')
ax4.scatter(b0['T_over_W'], np.zeros(len(b0)), color=BAND_COLORS[0], s=80, zorder=5, label=f'Band 0 non-liftoff (n={len(b0)})')
for t in range(N_TIERS):
    tt = b1[b1['h_max_tier'] == t]
    ax4.scatter(tt['T_over_W'], tt['h_max_m'], color=TIER_COLORS[t], s=40, zorder=4, label=f'Band 1 tier {t} (n={len(tt)})')
ax4.axvline(1.0, color='white', lw=1.2, ls='--', label='T/W = 1  (liftoff threshold)')
ax4.set_xlabel('T/W ratio at launch')
ax4.set_ylabel('h_max [m]')
ax4.set_title('Full selection in (T/W, h_max) space')
ax4.legend(fontsize=8, ncol=3)

fig.suptitle('NB7 · Unified 100-prop selection coverage', fontsize=14, fontweight='bold')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(PLOTS_DIR / '07_selection_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

### 4.6 Save Outputs

In [ ]:
SAVE_COLS = [
    'config_id', 'band', 'h_max_tier', 'subset',
    'radius_mm', 'blade_count',
    'inner_chord_mm', 'mid_chord_mm', 'outer_chord_mm',
    'inner_angle_deg', 'mid_angle_deg', 'outer_angle_deg',
    'mid_radial_pos',
    'mass_g', 'mass_kg', 'izz',
    'T_static_N', 'T_over_W', 'h_max_m', 'flight_time_s', 'v_impact_m_s',
]

save_columns = []
for column in SAVE_COLS:
    if column in selected.columns:
        save_columns.append(column)
out_df = selected[save_columns].copy()
out_df = out_df.sort_values(['band', 'h_max_tier', 'config_id']).reset_index(drop=True)

out_df.to_csv(OUT_CSV, index=False)
out_df.to_csv(OUT_ALL, index=False)

print(f'Saved: {OUT_CSV.name}  ({len(out_df)} rows)')
print(f'Saved: {OUT_ALL.name}  ({len(out_df)} rows)')
print()
print(out_df.groupby('subset').agg(
    n=('config_id', 'count'),
    T_over_W_min=('T_over_W', 'min'),
    T_over_W_max=('T_over_W', 'max'),
    h_max_min=('h_max_m', 'min'),
    h_max_max=('h_max_m', 'max'),
).round(3).to_string())

### 4.7 Copy Selected STLs

Copies the STL files of the 100 selected propellers into `representative_stl/` for printing and testing. The folder is cleared first so it always matches the current selection.


In [ ]:
STL_SRC_DIR = ROOT / 'stl'
STL_OUT_DIR = ROOT / 'representative_stl'
STL_OUT_DIR.mkdir(exist_ok=True)

for old_file in STL_OUT_DIR.glob('*.stl'):
    old_file.unlink()

copied = []
missing = []
for cid in out_df['config_id']:
    src = STL_SRC_DIR / f'prop_{int(cid)}.stl'
    dst = STL_OUT_DIR / f'prop_{int(cid)}.stl'
    if src.exists():
        shutil.copy2(src, dst)
        copied.append(int(cid))
    else:
        missing.append(int(cid))

print(f'STLs copied : {len(copied)} → {STL_OUT_DIR}')
if missing:
    print(f'STLs missing: {len(missing)} config_ids: {missing}')
else:
    print('All STLs found and copied successfully.')